# 🎬 hermes-acp-sdk — full showcase (live Hermes)

This notebook was **run against a real Hermes Agent** (`hermes-agent 0.18.2` on DeepSeek)
and is committed **with its output saved** — so you can read the real conversation, the
agent's reasoning and its token usage **without a key and without spending anything**.

**The tour**
1. Connect to a live agent and stream its answer
2. Watch it **think** — the reasoning stream
3. The full event taxonomy
4. The trap this SDK hides (and why nothing works without it)
5. Your own functions as agent tools (MCP)
6. Safe by default

## 0 · Setup

In [1]:
import os
from pathlib import Path

from hermes_acp_sdk import (
    HermesClient, AgentText, AgentThought, ToolCall, PlanUpdated,
    Usage, PermissionDenied, Finished, ToolServer,
)

# convenience: pick provider keys up from a nearby .env
for d in [Path.cwd(), *Path.cwd().parents]:
    f = d / ".env"
    if f.is_file():
        for line in f.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        break

# Hermes auto-detects the provider from whichever key is in the environment.
# This run used a FREE reasoning model on OpenRouter, so the reasoning stream below
# is real. Pass model=... to HermesClient to pin any model the agent advertises.
MODEL = "openrouter:tencent/hy3:free" if os.environ.get("OPENROUTER_API_KEY") else None

keys = [k for k in ("OPENROUTER_API_KEY", "DEEPSEEK_API_KEY", "GEMINI_API_KEY") if os.environ.get(k)]
print("provider key(s):", ", ".join(keys) or "none")
print("model          :", MODEL or "(agent default)")
print("HermesClient() finds the `hermes` binary itself — even in a venv that isn't activated.")

provider key(s): OPENROUTER_API_KEY
model          : openrouter:tencent/hy3:free
HermesClient() finds the `hermes` binary itself — even in a venv that isn't activated.


## 1 · Talk to a live Hermes — this is the whole API

`HermesClient()` spawns `hermes acp` and does the handshake. `session()` opens a session
**and selects a model**. `prompt()` is an async iterator, so streaming is a `for` loop.

In [2]:
async with HermesClient(model=MODEL) as hermes:
    print(f"connected to: {hermes.agent_name} {hermes.agent_version}\n")

    async with hermes.session() as s:
        async for ev in s.prompt("In one short sentence: what is a Python traceback?"):
            if isinstance(ev, AgentText):
                print(ev.text, end="", flush=True)          # streams in token by token
            elif isinstance(ev, Finished):
                print(f"\n\n[finished · stop_reason={ev.stop_reason}]")

connected to: hermes-agent 0.18.2



A

 Python traceback

 is a report

 that shows

...

 and

 why an

 exception was raised

.



[finished · stop_reason=end_turn]


## 2 · 🧠 Watch it think, and see every event

Everything the agent emits becomes a typed event. You decide what to render: the answer,
the reasoning, tool calls, the plan, token usage.

In [3]:
from collections import Counter

events = []
async with HermesClient(model=MODEL) as hermes:
    async with hermes.session() as s:
        print("models this agent offers:")
        for model_id, _name in s.available_models:
            print("   -", model_id)
        print()
        async for ev in s.prompt("Think it through briefly, then answer: what is 12 * 12?"):
            events.append(ev)

print("events received:", dict(Counter(type(e).__name__ for e in events)), "\n")

thoughts = "".join(e.text for e in events if isinstance(e, AgentThought))
answer   = "".join(e.text for e in events if isinstance(e, AgentText))
usage    = [e for e in events if isinstance(e, Usage)]

if thoughts:
    print("🧠 THOUGHTS >>>", thoughts[:220].strip(), "…\n")
else:
    print("🧠 THOUGHTS >>> (this model doesn't stream its reasoning — reasoning models like")
    print("               deepseek-reasoner do, and it arrives as AgentThought events)\n")
print("💬 ANSWER   >>>", answer.strip())
print("📊 USAGE    >>>", usage[-1] if usage else "n/a")

models this agent offers:
   - openrouter:anthropic/claude-fable-5
   - openrouter:anthropic/claude-opus-4.8
   - openrouter:anthropic/claude-opus-4.8-fast
   - openrouter:anthropic/claude-sonnet-5
   - openrouter:anthropic/claude-haiku-4.5
   - openrouter:openai/gpt-5.6-sol
   - openrouter:openai/gpt-5.6-sol-pro
   - openrouter:openai/gpt-5.6-terra
   - openrouter:openai/gpt-5.6-terra-pro
   - openrouter:openai/gpt-5.6-luna
   - openrouter:openai/gpt-5.6-luna-pro
   - openrouter:openai/gpt-5.5
   - openrouter:openai/gpt-5.5-pro
   - openrouter:openai/gpt-5.4-mini
   - openrouter:google/gemini-3.1-pro-preview
   - openrouter:google/gemini-3.5-flash
   - openrouter:x-ai/grok-4.5
   - openrouter:deepseek/deepseek-v4-pro
   - openrouter:deepseek/deepseek-v4-flash
   - openrouter:qwen/qwen3.7-max
   - openrouter:qwen/qwen3.7-plus
   - openrouter:qwen/qwen3.6-35b-a3b
   - openrouter:moonshotai/kimi-k2.6
   - openrouter:moonshotai/kimi-k2.7-code
   - openrouter:minimax/minimax-m3
   - openro

events received: {'Usage': 3, 'AgentThought': 15, 'AgentText': 4, 'Finished': 1} 

🧠 THOUGHTS >>> The user is asking a simple math question. This is a straightforward calculation. Let me just answer it directly.

12 * 12 = 144 …

💬 ANSWER   >>> 12 × 12 = **144**.
📊 USAGE    >>> Usage(input_tokens=12362, output_tokens=41, total_tokens=12403, used=None, size=None)


## 3 · 🪤 The trap this SDK hides for you

Two things documented **nowhere**, found by driving a real Hermes:

1. **`new_session` lies about the model.** It reports a `current_model_id`, but inference
   still goes out with an **empty** model, so *every* prompt comes back as
   `HTTP 400: The supported API model names are deepseek-v4-pro or deepseek-v4-flash, but you passed .`
   You **must** call `set_session_model` explicitly.
   **`session()` always does it for you** — that is the single biggest reason this package exists.
2. **Never pass `--provider` / `-m`** to `hermes` — those flags *blank out* the model.

The answers you just read are the proof: without the automatic `set_session_model` you
would have got the HTTP 400 blurb instead.

## 4 · 🛠 Give the agent your app's own tools (MCP)

`session(tools=[...])` runs an MCP server **inside this process**, so your functions keep
full access to your app's state. Here we serve one and call it with a real MCP client —
**no LLM involved, zero tokens.**

In [4]:
from mcp.client.session import ClientSession
from mcp.client.streamable_http import streamablehttp_client

ERROR_HISTORY = {"bex": "IndexError (9x), NameError (3x)"}

def student_weakness(student_id: str) -> str:
    "Look up which Python errors a given student most often gets wrong."
    return ERROR_HISTORY.get(student_id, "no history")

async with ToolServer([student_weakness], name="app-tools") as tools:
    cfg = tools.mcp_config
    print("MCP server :", cfg.url, "(loopback + bearer token)")
    headers = {h.name: h.value for h in cfg.headers}
    async with streamablehttp_client(cfg.url, headers=headers) as (r, w, _):
        async with ClientSession(r, w) as mcp:
            await mcp.initialize()
            listed = await mcp.list_tools()
            print("tool       :", listed.tools[0].name, "—", listed.tools[0].description)
            out = await mcp.call_tool("student_weakness", {"student_id": "bex"})
            print("CALL RESULT:", out.content[0].text)

[07/13/26 14:23:38] INFO     StreamableHTTP session manager started                  streamable_http_manager.py:131

MCP server : http://127.0.0.1:52244/mcp (loopback + bearer token)


                    INFO     Terminating session: None                                       streamable_http.py:788

                    INFO     HTTP Request: POST http://127.0.0.1:52244/mcp "HTTP/1.1 200 OK"        _client.py:1740

                    INFO     Negotiated protocol version: 2025-11-25                         streamable_http.py:193

...

                    INFO     Processing request of type CallToolRequest                               server.py:733

                    INFO     Terminating session: None                                       streamable_http.py:788

                    INFO     HTTP Request: POST http://127.0.0.1:52244/mcp "HTTP/1.1 200 OK"        _client.py:1740

CALL RESULT: IndexError (9x), NameError (3x)


                    INFO     StreamableHTTP session manager shutting down            streamable_http_manager.py:135

A real Hermes picks these up. Its own log during an ACP session reads:

```
MCP server 'app-tools' (HTTP): registered 5 tool(s): mcp__app_tools__student_weakness, ...
refreshed tool surface after ACP MCP registration (28 tools)
```

**This is the piece that lets a coach remember a student:** the agent can pull live
application state — a student's error history — through a function running in *your* process.

## 5 · 🔒 Safe by default *(zero tokens)*

In [5]:
from hermes_acp_sdk import EventHandler, DenyAll, FsPolicy, TerminalPolicy

handler = EventHandler(DenyAll(), FsPolicy(), TerminalPolicy())    # the defaults

for label, call in [
    ("read /etc/passwd", handler.read_text_file(path="/etc/passwd", session_id="s")),
    ("run `rm -rf /`",   handler.create_terminal(command="rm", session_id="s", args=["-rf", "/"])),
]:
    try:
        await call
    except PermissionError as e:
        print(f"❌ agent tried to {label:18} → {e}")

print("\nThese policies govern what the agent asks the CLIENT to do.")
print("They do NOT sandbox Hermes itself — the real boundary is the cwd you hand it.")

❌ agent tried to read /etc/passwd   → read access to '/etc/passwd' is not allowed
❌ agent tried to run `rm -rf /`     → terminal access is not allowed (TerminalPolicy.enabled is False)

These policies govern what the agent asks the CLIENT to do.
They do NOT sandbox Hermes itself — the real boundary is the cwd you hand it.


---
### Summary

`hermes-acp-sdk` turns a two-way protocol with a 12-method callback surface into
**one `async for` loop over typed events** — with the model-selection trap handled,
permissions/filesystem/terminals locked down by default, and your own Python functions
available to the agent as tools.

**No Jupyter required** — see `examples/chat.py` for the same API as a terminal chat app.

`pip install hermes-acp-sdk` · <https://github.com/VoixKz/hermes-acp-sdk>